# 02 · De grabación cruda a dataset de LeRobot

**Por qué convertir.** Cada librería de políticas quiere los datos en su
formato. LeRobot es el formato común: ACT, Diffusion Policy y los VLAs que
vamos a probar lo leen igual, y las herramientas de evaluación también.

Lo que cambia al convertir:
- los números pasan de un JSON por episodio a tablas Parquet;
- las imágenes pasan de JPEG sueltos a **video** (AV1), que pesa mucho menos;
- se calculan las **estadísticas** (media, desvío, etc.) que después usa la
  normalización.

Convertimos 3 episodios para mirar el resultado de cerca. La conversión
completa se corre igual, desde la terminal (ver `toolkit/README.md`).

In [ ]:
# torch first: torchvision and torch._dynamo have a circular import that only
# resolves if torch is loaded before lerobot.
import torch  # noqa: F401
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from toolkit.paths import DATASETS_DIR, RAW_RECORDINGS_DIR

plt.rcParams["figure.dpi"] = 110

In [ ]:
from toolkit.dataset.convert_unitree import ConvertArgs, convert

TASK_DIR = RAW_RECORDINGS_DIR / "pick_mameluco"
DEMO_ROOT = DATASETS_DIR / "local" / "nb_demo"
dataset = convert(ConvertArgs(raw_dir=TASK_DIR, repo_id="local/nb_demo", max_episodes=3, overwrite=True))

## Cómo queda en disco

In [ ]:
for path in sorted(DEMO_ROOT.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1e6:7.2f} MB  {path.relative_to(DEMO_ROOT)}")

- `meta/info.json`: qué hay (features, FPS, cantidad de episodios).
- `meta/stats.json`: estadísticas de todo el dataset.
- `meta/episodes/`: una fila por episodio, con sus propias estadísticas.
- `data/`: los números, cuadro por cuadro.
- `videos/`: la cámara.

In [ ]:
info = json.loads((DEMO_ROOT / "meta" / "info.json").read_text())
{k: {"dtype": v["dtype"], "shape": v["shape"]} for k, v in info["features"].items()}

In [ ]:
table = pd.concat(pd.read_parquet(p) for p in sorted((DEMO_ROOT / "data").rglob("*.parquet")))
table.head()

## Leerlo como lo va a leer el entrenamiento

`LeRobotDataset` devuelve un cuadro como diccionario de tensores. En la Mac
usamos `video_backend="pyav"` porque el decodificador por defecto necesita un
FFmpeg para arm64 que no está instalado.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("local/nb_demo", root=DEMO_ROOT, video_backend="pyav")
sample = ds[100]
{k: (tuple(v.shape), str(v.dtype)) if hasattr(v, "shape") else v for k, v in sample.items()}

La imagen ahora es `(3, 360, 640)` en float entre 0 y 1: canales primero, como
le gusta a PyTorch.

In [ ]:
plt.imshow(sample["observation.images.cam_right_high"].permute(1, 2, 0)); plt.axis("off");

## Cómo pide los datos ACT

ACT no predice una acción: predice las **próximas 48**. LeRobot lo resuelve con
`delta_timestamps`: le decís qué instantes relativos querés y te arma la
ventana.

In [ ]:
fps = ds.fps
ds_chunk = LeRobotDataset(
    "local/nb_demo", root=DEMO_ROOT, video_backend="pyav",
    delta_timestamps={"action": [k / fps for k in range(48)]},
)
chunk = ds_chunk[40]
print("action:", tuple(chunk["action"].shape), "  padding:", chunk["action_is_pad"].sum().item(), "pasos")

In [ ]:
j = 3  # codo izquierdo
plt.figure(figsize=(9, 3))
episode0_actions = np.stack(table[table["episode_index"] == 0]["action"].to_numpy())
plt.plot(episode0_actions[:, j], color="0.7", label="acción a lo largo del episodio")
plt.plot(range(40, 88), chunk["action"][:, j], lw=3, label="lo que ACT tiene que predecir desde t=40")
plt.legend(); plt.xlabel("cuadro"); plt.tight_layout()

## ¿La conversión perdió algo?

Comparamos contra el crudo. Los números tienen que ser **idénticos**. Las
imágenes no, porque el video comprime con pérdida: medimos cuánto.

In [ ]:
from toolkit.dataset.convert_unitree import SingleTaskJsonDataset
from toolkit.dataset.robot_configs import G1_BRAINCO_1CAM

raw_item = SingleTaskJsonDataset(TASK_DIR, G1_BRAINCO_1CAM).get_item(0)
ep0 = table[table["episode_index"] == 0]
state_conv = np.stack(ep0["observation.state"].to_numpy())
print("diferencia máxima en state:", np.abs(state_conv - raw_item["state"]).max())

k = 200
raw_img = raw_item["cameras"]["cam_right_high"][k].astype(np.float32) / 255
conv_img = ds[k]["observation.images.cam_right_high"].permute(1, 2, 0).numpy()
err = np.abs(raw_img - conv_img)
print(f"error medio por pixel: {err.mean() * 255:.2f} niveles de 255 (máx {err.max() * 255:.0f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
axes[0].imshow(raw_img); axes[0].set_title("JPEG crudo")
axes[1].imshow(conv_img); axes[1].set_title("después del video")
axes[2].imshow(err.mean(-1), cmap="magma"); axes[2].set_title("diferencia")
for ax in axes: ax.axis("off")
plt.tight_layout()

Los números quedan idénticos y la imagen cambia unos pocos niveles de gris por
pixel, sobre todo en bordes. Es el precio de pasar de 2,2 GB a ~0,3 GB.